# Imports

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score
import numpy as np

# Constants

In [2]:
batch_size = 1
device = 'cuda'

# Dataset

In [3]:
class ProbDataset(Dataset):
    def __init__(self, X: torch.tensor, Y: torch.tensor, bin_count, bin_width):
        super().__init__()
        self.X = X
        self.Y = Y
        self.bin_count = bin_count
        self.bin_width = bin_width
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index], self.Y[index]

In [4]:
train_dataset = torch.load('data/probabilistic/boiling_point_K_train.pt')
test_dataset = torch.load('data/probabilistic/boiling_point_K_test.pt')

C:\Users\agile\AppData\Local\Temp\ipykernel_12564\2287185075.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load('data/probabilistic/boiling_point

In [5]:
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Prerequisite Calculations

In [6]:
x_mean = train_dataset.X.mean(dim=0, keepdim=True)
x_std  = train_dataset.X.std(dim=0, keepdim=True)

train_dataset.X = (train_dataset.X - x_mean) / x_std
test_dataset.X = (test_dataset.X - x_mean) / x_std

# Model

In [ ]:
class RBFInterpolator:
    def __init__(self, X, Y, sigma=4.0):
        self.X = np.array(X)
        self.Y = np.array(Y)
        self.sigma = sigma

    def _gaussian_kernel(self, r2):
        return np.exp(-r2 / (2 * self.sigma ** 2))

    def predict(self, x):
        x = np.array(x)

        # compute squared distances to all points
        diff = self.X - x  # (N, D)
        r2 = np.sum(diff**2, axis=1)  # (N,)

        # compute weights
        weights = self._gaussian_kernel(r2)  # (N,)

        # avoid division by zero
        if np.sum(weights) == 0:
            return np.zeros(self.Y.shape[1])

        # normalize weights
        weights /= np.sum(weights)

        

        # weighted sum of outputs
        y = weights @ self.Y  # (M,)

        return y

In [8]:
X_train, Y_train = train_dataset.X.detach().cpu().numpy(), train_dataset.Y.detach().cpu().numpy()
X_test, Y_test = test_dataset.X.detach().cpu().numpy(), test_dataset.Y.detach().cpu().numpy()

In [ ]:
def evaluate_rbf(rbf, X_test, Y_test):
    preds = []

    for x in X_test:
        y_pred = rbf.predict(x)
        preds.append(y_pred)

    preds = np.array(preds)

    # R² score (multi-output)
    r2 = r2_score(Y_test, preds, multioutput='uniform_average')

    return r2, preds

rbf = RBFInterpolator(X_train, Y_train, sigma=0.001)
r2, preds = evaluate_rbf(rbf, X_test, Y_test)
print(r2)

0.8681388012618299
